In [5]:
import os
from tqdm import tqdm
import sys
if not hasattr(sys.modules[__name__], "cwd_changed"):
    os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(__name__))))
    sys.modules[__name__].cwd_changed = True

import warnings 
warnings.filterwarnings("ignore")
import logging
logging.getLogger('pgmpy').setLevel(logging.WARNING)


import pandas as pd
from utils.graph import dag_to_cpdag
# from metrics.graph import compare_dags
from utils.results import *


In [6]:
import pickle

subset = 'binary_data'
# subset = 'toyMedium'

def load_results(subset, base_dir= "results", ):
    global results_by_label

    with open(f'{base_dir}/pc/{subset}.pkl', 'rb') as f:
        results_pc = pickle.load(f)
    
    with open(f'{base_dir}/ges/{subset}.pkl', 'rb') as f:
        results_ges = pickle.load(f)

    with open(f'{base_dir}/hc/{subset}.pkl', 'rb') as f:
        results_hc = pickle.load(f)

    with open(f'{base_dir}/ea_fg/{subset}.pkl', 'rb') as f:
        results_fg = pickle.load(f)

    # with open(f'results/ea_ues/{subset}.pkl', 'rb') as f:
    #     results_ea_ues = pickle.load(f)

    # with open(f'results/ea_ies/{subset}.pkl', 'rb') as f:
    #     results_ea_ies = pickle.load(f)
    
    # with open(f'results/ea_fes/{subset}.pkl', 'rb') as f:
    #     results_ea_fes = pickle.load(f)

    # with open(f'results/ea_upu/{subset}.pkl', 'rb') as f:
    #     results_ea_upu = pickle.load(f)
    # with open(f'results/ea_ipu/{subset}.pkl', 'rb') as f:
    #     results_ea_ipu = pickle.load(f)
    

    results_by_label = {
        "PC": results_pc,
        "GES": results_ges,
        "HC": results_hc,
        # "EA (Informed)": results_ea_ies,
        "EA-FG": results_fg,
        # "HC Informed": results_hc_inf,
        # "EA Uninformed": results_ea_ues,
        # "EA Fully Informed": results_ea_fes,
        # "EA Uninformed (PU)": results_ea_upu,
        # "EA Informed (PU)": results_ea_ipu,
    }
    
load_results(subset)
# x_col="samples"
# y_col="p_noise"
# # filter = ("samples", 10_000)
# filter = ("n_xor", 10)
# filter_col, filter_val = filter    

In [8]:
import numpy as np
import pandas as pd

# ---------------------------
# Helpers: turn res.eval_metrics into 1-row DataFrame
# ---------------------------
def _eval_to_df(eval_metrics):
    """
    Accepts eval_metrics as DataFrame / Series / dict-like.
    Returns a DataFrame (possibly multi-row, but usually 1 row).
    """
    if eval_metrics is None:
        return None
    if isinstance(eval_metrics, pd.DataFrame):
        return eval_metrics.copy()
    if isinstance(eval_metrics, pd.Series):
        return eval_metrics.to_frame().T
    if isinstance(eval_metrics, dict):
        return pd.DataFrame([eval_metrics])
    # fallback: try to coerce
    try:
        return pd.DataFrame(eval_metrics)
    except Exception:
        return None


def build_long_metrics(results_by_label):
    """
    Stacks all eval_metrics rows across algorithms into one DataFrame.
    Adds Algorithm + run_id columns.
    """
    rows = []
    for alg, results_list in results_by_label.items():
        for i, res in enumerate(results_list):
            em = getattr(res, "eval_metrics", None)
            df_em = _eval_to_df(em)
            if df_em is None or len(df_em) == 0:
                continue

            df_em = df_em.copy()
            df_em["Algorithm"] = alg
            df_em["run_id"] = i
            rows.append(df_em)

    if not rows:
        raise ValueError("No eval_metrics found in results_by_label.")
    long_df = pd.concat(rows, ignore_index=True)
    return long_df


# ---------------------------
# Main: make "Median (IQR)" pivot table with bold best per (n_vars, samples)
# ---------------------------
def vstruct_precision_pivot(
    long_df: pd.DataFrame,
    metric_col: str = "Precision (Collider)",
    n_vars_col: str = "n_nodes",
    samples_col: str = "samples",
    algorithm_col: str = "Algorithm",
    extra_row_cols=(),   # <--- NEW
    filter_tuples=None,
    alg_order=None,
    decimals: int = 2,
):
    df = long_df.copy()

    # optional filtering
    if filter_tuples is not None:
        for fcol, fval in filter_tuples:
            df = df[df[fcol] == fval].copy()

    # Ensure numeric types
    df[n_vars_col] = df[n_vars_col].astype(int)
    df[samples_col] = df[samples_col].astype(int)
    for c in extra_row_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="ignore")

    df[metric_col] = pd.to_numeric(df[metric_col], errors="coerce")
    df = df.dropna(subset=[metric_col])

    row_cols = list(extra_row_cols) + [n_vars_col, algorithm_col]
    g = (
        df.groupby(row_cols + [samples_col])[metric_col]
          .agg(
              median="median",
              q25=lambda s: s.quantile(0.25),
              q75=lambda s: s.quantile(0.75),
          )
          .reset_index()
    )
    g["iqr"] = g["q75"] - g["q25"]
    g["cell"] = g.apply(
        lambda r: f"{r['median']:.{decimals}f} ({r['iqr']:.{decimals}f})",
        axis=1
    )

    table_txt = g.pivot(index=row_cols, columns=samples_col, values="cell")
    table_med = g.pivot(index=row_cols, columns=samples_col, values="median")

    # pretty sample headers like 5,000
    table_txt = table_txt.rename(columns=lambda x: f"{int(x):,}")
    table_med = table_med.rename(columns=lambda x: f"{int(x):,}")

    # algorithm ordering (now algorithm is the LAST row level)
    alg_level = len(row_cols) - 1
    if alg_order is not None:
        table_txt = table_txt.reindex(alg_order, level=alg_level)
        table_med = table_med.reindex(alg_order, level=alg_level)

    # bold best median per (p_noise, n_xor, n_vars, sample) across algorithms
    group_levels = list(range(len(row_cols) - 1))  # all row levels except algorithm
    is_best = table_med.groupby(level=group_levels).transform(lambda x: x == x.max())

    table_bold = table_txt.copy()
    for col in table_bold.columns:
        mask = is_best[col].fillna(False)
        table_bold.loc[mask, col] = table_bold.loc[mask, col].apply(lambda x: f"\\textbf{{{x}}}")

    return table_bold


# ---------------------------
# Usage with your current setup
# ---------------------------
long_df = build_long_metrics(results_by_label)


In [10]:
long_df.to_csv('results/long_metrics.csv', index=False)

In [12]:

# Your filter tuple
tab = vstruct_precision_pivot(
    long_df,
    metric_col="Recall [Synergy]",
    n_vars_col="n_nodes",
    samples_col="samples",
    # extra_row_cols=(),
    alg_order=list(results_by_label.keys()),
    filter_tuples=[("p_noise", 0.1), ("dag_family", "ba")],  # don't filter n_xor anymore if you want it in the table
)

# Print table in notebook
display(tab)

# Export to LaTeX (escape=False so \textbf works)
latex = tab.to_latex(
    escape=False,
    caption="Pivot table of V-structure precision. Values are Median (IQR). Best per (variables, samples) in bold.",
    label="tab:vstruct_precision",
)
print(latex)


samples                             500                 1,000  \
n_nodes Algorithm                                               
10      PC                  0.00 (0.00)           0.00 (0.00)   
        GES                 0.00 (0.00)           0.00 (0.00)   
        HC                  0.00 (0.00)           0.00 (0.00)   
        EA-FG      \textbf{1.00 (0.75)}  \textbf{1.00 (0.75)}   
19      GES        \textbf{0.00 (0.00)}  \textbf{0.00 (0.00)}   
        HC         \textbf{0.00 (0.00)}  \textbf{0.00 (0.00)}   
20      PC                  0.00 (0.00)           0.00 (0.00)   
        GES                         NaN           0.00 (0.00)   
        HC                          NaN           0.00 (0.00)   
        EA-FG      \textbf{1.00 (0.75)}  \textbf{1.00 (0.75)}   
27      GES        \textbf{0.00 (0.00)}  \textbf{0.00 (0.00)}   
        HC         \textbf{0.00 (0.00)}  \textbf{0.00 (0.00)}   
28      GES        \textbf{0.00 (0.00)}                   NaN   
        HC         \textbf{0.00 (0.00)}                   NaN   
29      GES                         NaN  \textbf{0.00 (0.00)}   
        HC                          NaN  \textbf{0.00 (0.00)}   
30      PC                  0.00 (0.00)           0.00 (0.00)   
        EA-FG      \textbf{1.00 (0.75)}  \textbf{1.00 (0.75)}   

samples                           2,000                 5,000  \
n_nodes Algorithm                                               
10      PC                  0.00 (0.00)           0.00 (0.00)   
        GES                 0.00 (0.00)           0.00 (0.00)   
        HC                  0.00 (0.00)           0.00 (0.00)   
        EA-FG      \textbf{1.00 (0.75)}  \textbf{1.00 (0.75)}   
19      GES        \textbf{0.00 (0.00)}  \textbf{0.00 (0.00)}   
        HC         \textbf{0.00 (0.00)}  \textbf{0.00 (0.00)}   
20      PC                  0.00 (0.00)           0.00 (0.00)   
        GES                         NaN                   NaN   
        HC                          NaN                   NaN   
        EA-FG      \textbf{1.00 (0.75)}  \textbf{1.00 (0.75)}   
27      GES        \textbf{0.00 (0.00)}  \textbf{0.00 (0.12)}   
        HC         \textbf{0.00 (0.00)}  \textbf{0.00 (0.38)}   
28      GES        \textbf{0.00 (0.00)}                   NaN   
        HC         \textbf{0.00 (0.00)}                   NaN   
29      GES        \textbf{0.00 (0.00)}                   NaN   
        HC         \textbf{0.00 (0.00)}                   NaN   
30      PC                  0.00 (0.00)           0.00 (0.00)   
        EA-FG      \textbf{1.00 (0.75)}  \textbf{1.00 (0.75)}   

samples                          10,000  
n_nodes Algorithm                        
10      PC                  0.00 (0.00)  
        GES                 0.00 (0.00)  
        HC                  0.00 (0.00)  
        EA-FG      \textbf{1.00 (0.75)}  
19      GES        \textbf{0.00 (0.00)}  
        HC         \textbf{0.00 (0.00)}  
20      PC                  0.00 (0.00)  
        GES                         NaN  
        HC                          NaN  
        EA-FG      \textbf{1.00 (0.75)}  
27      GES        \textbf{0.00 (0.00)}  
        HC         \textbf{0.00 (0.00)}  
28      GES        \textbf{0.00 (0.00)}  
        HC         \textbf{0.00 (0.00)}  
29      GES                         NaN  
        HC                          NaN  
30      PC                  0.00 (0.00)  
        EA-FG      \textbf{1.00 (0.75)}

\begin{table}
\caption{Pivot table of V-structure precision. Values are Median (IQR). Best per (variables, samples) in bold.}
\label{tab:vstruct_precision}
\begin{tabular}{lllllll}
\toprule
 & samples & 500 & 1,000 & 2,000 & 5,000 & 10,000 \\
n_nodes & Algorithm &  &  &  &  &  \\
\midrule
\multirow[t]{4}{*}{10} & PC & 0.00 (0.00) & 0.00 (0.00) & 0.00 (0.00) & 0.00 (0.00) & 0.00 (0.00) \\
 & GES & 0.00 (0.00) & 0.00 (0.00) & 0.00 (0.00) & 0.00 (0.00) & 0.00 (0.00) \\
 & HC & 0.00 (0.00) & 0.00 (0.00) & 0.00 (0.00) & 0.00 (0.00) & 0.00 (0.00) \\
 & EA-FG & \textbf{1.00 (0.75)} & \textbf{1.00 (0.75)} & \textbf{1.00 (0.75)} & \textbf{1.00 (0.75)} & \textbf{1.00 (0.75)} \\
\cline{1-7}
\multirow[t]{2}{*}{19} & GES & \textbf{0.00 (0.00)} & \textbf{0.00 (0.00)} & \textbf{0.00 (0.00)} & \textbf{0.00 (0.00)} & \textbf{0.00 (0.00)} \\
 & HC & \textbf{0.00 (0.00)} & \textbf{0.00 (0.00)} & \textbf{0.00 (0.00)} & \textbf{0.00 (0.00)} & \textbf{0.00 (0.00)} \\
\cline{1-7}
\multirow[t]{4}{*}{20} & PC 

In [6]:
# Your filter tuple
tab = vstruct_precision_pivot(
    long_df,
    metric_col="Recall (Synergy)",
    n_vars_col="n_nodes",
    samples_col="samples",
    # extra_row_cols=(),
    alg_order=list(results_by_label.keys()),
    filter_tuples=[("p_noise", 0.1), ("n_xor", 10)],  # don't filter n_xor anymore if you want it in the table
)

# Print table in notebook
display(tab)

# Export to LaTeX (escape=False so \textbf works)
latex = tab.to_latex(
    escape=False,
    caption="Pivot table of V-structure precision. Values are Median (IQR). Best per (variables, samples) in bold.",
    label="tab:vstruct_precision",
)
print(latex)

KeyError: 'n_xor'